# 06 — Baseline MostPopular trên 3 chiến lược chia tập

**Mục tiêu:** đo hiệu năng của mô hình "gợi ý phim phổ biến nhất" trên cả 3 split, làm sàn so sánh cho ALS ở notebook 08.

**Input:** split của notebook 04, audit của notebook 05 (cần `eval_users`).

**Output:** `runs/<run_id>/outputs/mostpopular_{split}.json`.

**Gate L6a:** Popularity tính **trên train** (không phải full data), tie-break đơn định, NDCG khớp kết quả đo thật.

## Khái niệm — vì sao baseline này quan trọng

MostPopular là mô hình rẻ nhất có thể tưởng tượng: đếm item nào được xem nhiều nhất trong train, đề xuất đúng danh sách đó cho mọi user. Vì nó không học gì từ sở thích cá nhân, nó là **sàn hiệu năng**.

Giá trị của nó trong đề tài này **không** phải để thấy nó yếu. Giá trị là:

1. Nếu MostPopular thắng ALS ở một split nào đó, ta biết split đó có vấn đề (mô hình không học được gì từ dữ liệu test).
2. Nó cho một điểm neo để đọc `popularity_bias_ratio`: một mô hình "cá nhân hóa" mà vẫn có bias gần bằng MostPopular thì thực chất chỉ là MostPopular đội lốt.

Hai quy tắc phải giữ tuyệt đối:

- **Popularity tính trên train.** Dùng full dataset để đếm là rò rỉ test vào baseline, làm sàn bị đẩy cao giả tạo.
- **Tie-break đơn định.** Rất nhiều item có cùng số tương tác; không tie-break thì top-10 thay đổi giữa các lần chạy và kết quả không tái lập được. Ở đây dùng `movieId` tăng dần.

**Về chỉ số thiên lệch:**

- `catalog_coverage`: bao nhiêu phần trăm catalog được đề xuất. MostPopular luôn chỉ đề xuất 10 item → coverage cực thấp. Đó là bản chất, không phải bug.
- `gini_exposure`: 0 là mọi item được đề xuất như nhau, → 1 là dồn hết vào một vài item. MostPopular gần 1.
- `popularity_bias_ratio` = popularity trung bình của item được đề xuất / popularity trung bình toàn catalog. MostPopular luôn rất cao vì nó chọn đúng item phổ biến nhất.

## 1. Nạp split và audit

In [1]:
from pathlib import Path
import json
import sys
import time

import numpy as np

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'src').is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from IPython import get_ipython
from src.recsys.notebook_logging import NotebookRun
from src.utils import spark_utils
from src.audit import splits as split_lib
from src.audit import leakage, popularity as pop_lib
from src.audit import metrics
from pyspark.sql import functions as F

run = NotebookRun(
    project_root,
    '06_mostpopular_baseline',
    metadata={
        'protocol_id': spark_utils.PROTOCOL_ID,
        'model': 'MostPopular',
        'k': spark_utils.TOP_K,
        'relevance_threshold': spark_utils.RELEVANCE_THRESHOLD,
        'min_train': spark_utils.MIN_TRAIN,
        'purpose': 'popularity floor across split strategies',
    },
)
run.attach_ipython(get_ipython())
print('Run ID:', run.run_id)

runs_dir = project_root / 'runs'
audit_candidates = sorted(runs_dir.glob('*/outputs/data_level_audit.json'), key=lambda p: p.stat().st_mtime, reverse=True)
assert audit_candidates, 'data_level_audit.json not found; run notebook 05 first.'
audit_path = audit_candidates[0]
audit_doc = json.loads(audit_path.read_text(encoding='utf-8'))
audits = audit_doc['audits']
assert set(audits) == set(split_lib.STRATEGIES)
print('Audit source:', audit_path)
print('Protocol:', audit_doc['protocol_id'])
run.checkpoint('audit_loaded', 'PASS', checks={'audit_path': str(audit_path), 'strategies': list(audits)})

Run ID: 20261115_233832_06_mostpopular_baseline_c56e4ebf
Audit source: D:\Master Exers\Big Data\Recomendation_System\runs\20261115_233616_05_audit_split_leakage_3fa5a49e\outputs\data_level_audit.json
Protocol: project_protocol_v1


WindowsPath('D:/Master Exers/Big Data/Recomendation_System/runs/20261115_233832_06_mostpopular_baseline_c56e4ebf/checkpoints/audit_loaded.json')

In [2]:
spark = spark_utils.get_spark('RecSys06MostPopular')
print('Spark:', spark.version, '| master:', spark.sparkContext.master)

C:\Users\PC\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark: 4.2.0 | master: local[4]


## 2. Popularity phải tính trên train

Ô dưới đây đếm lại số tương tác mỗi item từ train của từng split, in top-10. Chú ý top-10 thay đổi giữa các split — đó là bằng chứng popularity phụ thuộc vào cách chia, không phải hằng số của dataset.

In [3]:
topk_by_split = {}
popularity_by_split = {}

for strategy in split_lib.STRATEGIES:
    train_df, _ = split_lib.load_split(strategy, spark_utils.SPLITS_DIR)
    pop_map = pop_lib.train_popularity(train_df)
    topk = pop_lib.mostpopular_topk(train_df, k=spark_utils.TOP_K, popularity=pop_map)
    topk_by_split[strategy] = topk
    popularity_by_split[strategy] = pop_map

    print(f'--- {strategy} ---')
    print(f'  catalog in train: {len(pop_map):,} items')
    print(f'  top-{spark_utils.TOP_K}: {topk}')
    print(f'  top item count: {pop_map[topk[0]]:,}')
    run.log('POPULARITY_TOPK', strategy=strategy, topk=topk, catalog=len(pop_map))

print()
print('So sanh top-10 giua 3 split:')
for strategy in split_lib.STRATEGIES:
    print(f'  {strategy:<18}{topk_by_split[strategy]}')
run.checkpoint(
    'topk_computed',
    'PASS',
    checks={
        'catalog_sizes': {s: len(popularity_by_split[s]) for s in split_lib.STRATEGIES},
        'topk_lengths': {s: len(topk_by_split[s]) for s in split_lib.STRATEGIES},
    },
)

--- random80_20 ---
  catalog in train: 56,604 items
  top-10: [356, 318, 296, 593, 2571, 260, 480, 527, 110, 2959]
  top item count: 65,369
--- temporal_global ---
  catalog in train: 34,461 items
  top-10: [356, 296, 318, 593, 480, 260, 2571, 110, 527, 589]
  top item count: 67,781
--- leave_one_out ---
  catalog in train: 58,987 items
  top-10: [356, 318, 296, 593, 2571, 260, 480, 527, 110, 2959]
  top item count: 80,936

So sanh top-10 giua 3 split:
  random80_20       [356, 318, 296, 593, 2571, 260, 480, 527, 110, 2959]
  temporal_global   [356, 296, 318, 593, 480, 260, 2571, 110, 527, 589]
  leave_one_out     [356, 318, 296, 593, 2571, 260, 480, 527, 110, 2959]


WindowsPath('D:/Master Exers/Big Data/Recomendation_System/runs/20261115_233832_06_mostpopular_baseline_c56e4ebf/checkpoints/topk_computed.json')

## 3. NDCG và HitRate trên tập eval_users

Với MostPopular, mọi user nhận cùng 10 item, nên chỉ cần đếm item nào trong 10 item đó là relevant cho từng user. Code dưới dùng đúng `src.audit.metrics` mà notebook 07 sẽ kiểm tra tay — không viết công thức NDCG riêng ở đây.

In [4]:
metrics_by_split = {}

for strategy in split_lib.STRATEGIES:
    started = time.perf_counter()
    train_df, test_df = split_lib.load_split(strategy, spark_utils.SPLITS_DIR)
    eligible = leakage.eligible_users(train_df, spark_utils.MIN_TRAIN)

    truth_rows = (
        test_df.join(F.broadcast(eligible), 'userId')
        .filter(F.col('rating') >= F.lit(spark_utils.RELEVANCE_THRESHOLD))
        .groupBy('userId')
        .agg(F.collect_list('movieId').alias('relevant'))
        .collect()
    )
    truth = {int(r['userId']): {int(i) for i in r['relevant']} for r in truth_rows}

    topk = topk_by_split[strategy]
    recommendations = pop_lib.popularity_recommendations(topk, list(truth.keys()))

    ndcg_mean, ndcg_per_user = metrics.mean_metric(recommendations, truth, metrics.ndcg_at_k, spark_utils.TOP_K)
    dcg_mean, _ = metrics.mean_metric(recommendations, truth, metrics.dcg_at_k, spark_utils.TOP_K)
    hit_mean, _ = metrics.mean_metric(recommendations, truth, metrics.hit_at_k, spark_utils.TOP_K)
    recall_mean, _ = metrics.mean_metric(recommendations, truth, metrics.recall_at_k, spark_utils.TOP_K)

    metrics_by_split[strategy] = {
        'model': 'MostPopular',
        'strategy': strategy,
        'ndcg10': round(ndcg_mean, 4),
        'dcg10': round(dcg_mean, 4),
        'hit_rate10': round(hit_mean, 4),
        'recall10': round(recall_mean, 4),
        'eval_users': len(ndcg_per_user),
        'relevant_test_items': sum(len(v) for v in truth.values()),
        'topk': topk,
        'elapsed_seconds': round(time.perf_counter() - started, 1),
    }

    print(f'--- {strategy} ---')
    print(f"  ndcg@10={ndcg_mean:.4f}  dcg@10={dcg_mean:.4f}  hit@10={hit_mean:.4f}  recall@10={recall_mean:.4f}")
    print(f'  eval users={len(ndcg_per_user):,}  relevant test items={sum(len(v) for v in truth.values()):,}')
    run.log('POPULARITY_METRICS', strategy=strategy, metrics=metrics_by_split[strategy])

run.checkpoint('metrics_computed', 'PASS', checks={'strategies': list(metrics_by_split)})

--- random80_20 ---
  ndcg@10=0.0852  dcg@10=0.3197  hit@10=0.4546  recall@10=0.0712
  eval users=158,692  relevant test items=2,491,871
--- temporal_global ---
  ndcg@10=0.0455  dcg@10=0.1982  hit@10=0.2425  recall@10=0.0146
  eval users=6,219  relevant test items=290,588
--- leave_one_out ---
  ndcg@10=0.0183  dcg@10=0.0183  hit@10=0.0384  recall@10=0.0384
  eval users=96,003  relevant test items=96,003


WindowsPath('D:/Master Exers/Big Data/Recomendation_System/runs/20261115_233832_06_mostpopular_baseline_c56e4ebf/checkpoints/metrics_computed.json')

## 4. Chỉ số thiên lệch phổ biến

Với MostPopular, `longtail_coverage` sẽ bằng 0 vì nó chỉ đề xuất 10 item phổ biến nhất — đó là định nghĩa của baseline này, không phải lỗi. Con số này quan trọng khi **so với ALS** ở notebook 08: ALS có thực sự phân tán hơn, hay cũng dồn vào item phổ biến?

In [5]:
for strategy in split_lib.STRATEGIES:
    bias = pop_lib.bias_metrics(
        topk_by_split[strategy],
        popularity_by_split[strategy],
    )
    metrics_by_split[strategy]['bias'] = bias
    print(f'--- {strategy} ---')
    for key in ('catalog_coverage', 'longtail_coverage', 'gini_exposure',
                'avg_pop_rec', 'avg_pop_all', 'popularity_bias_ratio',
                'tail_threshold_pop', 'tail_items'):
        print(f'  {key:<26}{bias[key]}')
    run.log('POPULARITY_BIAS', strategy=strategy, bias=bias)

print()
print('Doc so: MostPopular chi de xuat', spark_utils.TOP_K, 'item nen coverage gan 0 va gini gan 1.')
print('popularity_bias_ratio hang chuc lan nghia la no chon dung nhom phim pho bien nhat.')

--- random80_20 ---
  catalog_coverage          0.000177
  longtail_coverage         0.0
  gini_exposure             0.9998
  avg_pop_rec               56019.5
  avg_pop_all               353.3
  popularity_bias_ratio     158.5625
  tail_threshold_pop        2
  tail_items                18701
--- temporal_global ---
  catalog_coverage          0.00029
  longtail_coverage         0.0
  gini_exposure             0.9997
  avg_pop_rec               58764.4
  avg_pop_all               580.35
  popularity_bias_ratio     101.2576
  tail_threshold_pop        2
  tail_items                9737
--- leave_one_out ---
  catalog_coverage          0.00017
  longtail_coverage         0.0
  gini_exposure             0.9998
  avg_pop_rec               69589.8
  avg_pop_all               421.07
  popularity_bias_ratio     165.2696
  tail_threshold_pop        2
  tail_items                17942

Doc so: MostPopular chi de xuat 10 item nen coverage gan 0 va gini gan 1.
popularity_bias_ratio hang chuc lan

## 5. Đối chiếu kết quả đo thật và ghi artifact

In [6]:
expected_ndcg = {'random80_20': 0.0852, 'temporal_global': 0.0455, 'leave_one_out': 0.0183}
mismatches = []
for strategy, want in expected_ndcg.items():
    got = metrics_by_split[strategy]['ndcg10']
    ok = abs(got - want) <= 0.003
    print(f"{'OK  ' if ok else 'DIFF'} {strategy:<18}ndcg@10 got={got:<8}want={want}")
    if not ok:
        mismatches.append((strategy, want, got))

print()
if mismatches:
    print('MISMATCH:', mismatches)
    print('Kiem tra: top-10 co doi khong (tie-break), relevance threshold, eval_users.')
else:
    print('NDCG khop ket qua do that.')

run.log('REPRODUCTION_CHECK', mismatches=len(mismatches), detail=mismatches)
run.checkpoint(
    'reproduction_check',
    'PASS' if not mismatches else 'FAIL',
    checks={'mismatches': len(mismatches), 'detail': mismatches},
    note='Tolerance 0.003: popularity counts depend on split partition layout.',
)

OK   random80_20       ndcg@10 got=0.0852  want=0.0852
OK   temporal_global   ndcg@10 got=0.0455  want=0.0455
OK   leave_one_out     ndcg@10 got=0.0183  want=0.0183

NDCG khop ket qua do that.


WindowsPath('D:/Master Exers/Big Data/Recomendation_System/runs/20261115_233832_06_mostpopular_baseline_c56e4ebf/checkpoints/reproduction_check.json')

In [7]:
outputs_dir = run.run_dir / 'outputs'
outputs_dir.mkdir(parents=True, exist_ok=True)

paths = {}
for strategy in split_lib.STRATEGIES:
    path = outputs_dir / f"mostpopular_{strategy}.json"
    path.write_text(
        json.dumps(metrics_by_split[strategy], ensure_ascii=False, indent=2, default=str),
        encoding='utf-8',
    )
    paths[strategy] = str(path)
    print('saved:', path)

summary = {
    'protocol_id': spark_utils.PROTOCOL_ID,
    'model': 'MostPopular',
    'k': spark_utils.TOP_K,
    'relevance_threshold': spark_utils.RELEVANCE_THRESHOLD,
    'results': {s: {k: v for k, v in metrics_by_split[s].items() if k != 'topk'} for s in split_lib.STRATEGIES},
    'mismatches': mismatches,
}
summary_path = outputs_dir / 'mostpopular_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

reloaded = json.loads(summary_path.read_text(encoding='utf-8'))
assert set(reloaded['results']) == set(split_lib.STRATEGIES)

print()
header = f"{'split':<18}{'ndcg@10':>10}{'hit@10':>10}{'gini':>9}{'bias_ratio':>12}{'eval_users':>12}"
print(header)
print('-' * len(header))
for strategy in split_lib.STRATEGIES:
    m = metrics_by_split[strategy]
    print(f"{strategy:<18}{m['ndcg10']:>10.4f}{m['hit_rate10']:>10.4f}"
          f"{m['bias']['gini_exposure']:>9.4f}{m['bias']['popularity_bias_ratio']:>12.2f}{m['eval_users']:>12,}")

run.checkpoint(
    'artifacts_saved',
    'PASS',
    checks={'files': len(paths), 'summary_readback': True, 'mismatches': len(mismatches)},
    note=str(summary_path),
)
run.finish('COMPLETED', summary={'model': 'MostPopular', 'summary': str(summary_path), 'mismatches': len(mismatches)})
spark_utils.stop_spark(spark)
print('L6a COMPLETE. Next: 07_ndcg_manual_check.ipynb')

saved: D:\Master Exers\Big Data\Recomendation_System\runs\20261115_233832_06_mostpopular_baseline_c56e4ebf\outputs\mostpopular_random80_20.json
saved: D:\Master Exers\Big Data\Recomendation_System\runs\20261115_233832_06_mostpopular_baseline_c56e4ebf\outputs\mostpopular_temporal_global.json
saved: D:\Master Exers\Big Data\Recomendation_System\runs\20261115_233832_06_mostpopular_baseline_c56e4ebf\outputs\mostpopular_leave_one_out.json

split                ndcg@10    hit@10     gini  bias_ratio  eval_users
-----------------------------------------------------------------------
random80_20           0.0852    0.4546   0.9998      158.56     158,692
temporal_global       0.0455    0.2425   0.9997      101.26       6,219
leave_one_out         0.0183    0.0384   0.9998      165.27      96,003
L6a COMPLETE. Next: 07_ndcg_manual_check.ipynb
